<a href="https://colab.research.google.com/github/contreras-juan/procesamiento_lenguaje_natural/blob/main/Cuadernos/07_Clasificacion_NLTK.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<h1 style="color: #FECB05; text-align: center;">Clasificación de texto con NLTK</h1>

<h2 style="color: #007ACC;">Autores</h2>

- [Juan Felipe Contreras Alcívar](https://www.linkedin.com/in/juanf-contreras/)

---

<h2 style="color: #007ACC;">Tabla de contenido</h2>

- [<span style="color: #005C99;">Introducción</span>](#introduccion)
- [<span style="color: #005C99;">La API de clasificación de NLTK</span>](#la-api-de-clasificacion-de-nltk)
- [<span style="color: #005C99;">Diseño iterativo de características</span>](#diseno-iterativo-de-caracteristicas)
- [<span style="color: #005C99;">Clasificador de máxima entropía</span>](#clasificador-de-maxima-entropia)
- [<span style="color: #005C99;">Clasificación de oraciones: subjetividad</span>](#clasificacion-de-oraciones-subjetividad)
- [<span style="color: #005C99;">Evaluación con nltk.metrics</span>](#evaluacion-con-nltk-metrics)
- [<span style="color: #005C99;">Integración con scikit-learn</span>](#integracion-con-scikit-learn)
- [<span style="color: #005C99;">Referencias</span>](#referencias)

---

<h2 style="color: #007ACC;">Instalación de librerías</h2>

Ejecuta esta celda si trabajas en Google Colab o en un entorno nuevo.

In [1]:
# Instalación de librerías necesarias para esta sesión
# (útil en Google Colab o en un entorno nuevo)
%pip install -q numpy pandas matplotlib scikit-learn nltk

Note: you may need to restart the kernel to use updated packages.


In [2]:
import random
import time
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import nltk
from nltk.corpus import names, subjectivity, stopwords
from nltk.classify import NaiveBayesClassifier, MaxentClassifier, accuracy
from nltk.classify.scikitlearn import SklearnClassifier
from nltk.metrics import ConfusionMatrix, precision, recall, f_measure

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

for recurso in ["names", "subjectivity", "stopwords"]:
    nltk.download(recurso, quiet=True)

<a id="introduccion"></a>
<h2 style="color: #007ACC;">Introducción</h2>

En el módulo 1 clasificamos texto con `scikit-learn`, donde cada documento es una fila de una matriz numérica. NLTK ofrece una interfaz distinta, pensada para el **diseño explícito de características**: cada ejemplo se describe con un **diccionario** de características con nombre.

```python
{"ultima_letra": "a", "longitud": 5, "contiene(excelente)": True}
```

Esta forma de trabajar es muy didáctica: obliga a pensar qué información lingüística es relevante para la tarea, y los clasificadores permiten inspeccionar fácilmente qué características usó el modelo.

En este cuaderno:

1. Aprendemos la API con un problema clásico: predecir el género asociado a un nombre propio.
2. Seguimos un ciclo de **diseño iterativo de características** con análisis de errores.
3. Conocemos el **clasificador de máxima entropía**, base de los MEMM del cuaderno 10.
4. Clasificamos oraciones subjetivas y objetivas, y evaluamos con `nltk.metrics`.

<a id="la-api-de-clasificacion-de-nltk"></a>
<h2 style="color: #007ACC;">La API de clasificación de NLTK</h2>

La API tiene tres piezas:

| Pieza | Qué es |
|-------|--------|
| **Extractor de características** | Una función que recibe un ejemplo y devuelve un `dict` |
| **Conjunto de entrenamiento** | Una lista de pares `(características, etiqueta)` |
| **Clasificador** | Se entrena con `Clasificador.train(...)` y predice con `.classify(...)` |

Usaremos el corpus `names`: unos 8 000 nombres en inglés etiquetados como masculinos o femeninos.

> Esta tarea es solo un ejemplo didáctico de ingeniería de características. Asociar un género a un nombre es una simplificación (hay nombres usados por personas de cualquier género) y no debe usarse para inferir el género de personas reales.

In [3]:
nombres = [(n, "masculino") for n in names.words("male.txt")] + [(n, "femenino") for n in names.words("female.txt")]
random.Random(SEED).shuffle(nombres)

print(f"Total: {len(nombres)} nombres")
print(Counter(g for _, g in nombres))
print(nombres[:8])

Total: 7944 nombres
Counter({'femenino': 5001, 'masculino': 2943})
[('Raye', 'femenino'), ('Marita', 'femenino'), ('Fey', 'femenino'), ('Vittoria', 'femenino'), ('Dannie', 'femenino'), ('Annabela', 'femenino'), ('Sayre', 'masculino'), ('Brock', 'masculino')]


Dividimos en tres conjuntos:

- **Entrenamiento:** para ajustar el modelo.
- **Desarrollo (dev):** para analizar errores y decidir qué características agregar.
- **Prueba:** solo para la evaluación final. Si lo usáramos para diseñar características, acabaríamos sobreajustándonos a él.

In [4]:
test_nombres, dev_nombres, train_nombres = nombres[:1000], nombres[1000:2000], nombres[2000:]
print(f"Entrenamiento: {len(train_nombres)} | Desarrollo: {len(dev_nombres)} | Prueba: {len(test_nombres)}")


def caracteristicas_v1(nombre):
    return {"ultima_letra": nombre[-1].lower()}


def evaluar(extractor, clasificador_cls=NaiveBayesClassifier, **kwargs):
    train_set = [(extractor(n), g) for n, g in train_nombres]
    dev_set = [(extractor(n), g) for n, g in dev_nombres]
    clf = clasificador_cls.train(train_set, **kwargs)
    return clf, accuracy(clf, dev_set)


clf_v1, acc_v1 = evaluar(caracteristicas_v1)
print(f"Exactitud en dev (última letra): {acc_v1:.3f}")
print(clf_v1.classify(caracteristicas_v1("Andrea")), clf_v1.classify(caracteristicas_v1("Gabriel")))

Entrenamiento: 5944 | Desarrollo: 1000 | Prueba: 1000
Exactitud en dev (última letra): 0.771
femenino femenino


Con **una sola característica** (la última letra) ya superamos ampliamente el azar. `show_most_informative_features` muestra las características con mayor razón de verosimilitud entre clases, la misma $\lambda(w)$ del cuaderno 04 (aquí sin logaritmo):

In [5]:
clf_v1.show_most_informative_features(8)

Most Informative Features
            ultima_letra = 'a'            femeni : mascul =     29.8 : 1.0
            ultima_letra = 'k'            mascul : femeni =     26.2 : 1.0
            ultima_letra = 'f'            mascul : femeni =     24.6 : 1.0
            ultima_letra = 'v'            mascul : femeni =     12.0 : 1.0
            ultima_letra = 'p'            mascul : femeni =     11.3 : 1.0
            ultima_letra = 'd'            mascul : femeni =      9.0 : 1.0
            ultima_letra = 'o'            mascul : femeni =      8.8 : 1.0
            ultima_letra = 'm'            mascul : femeni =      7.2 : 1.0


<a id="diseno-iterativo-de-caracteristicas"></a>
<h2 style="color: #007ACC;">Diseño iterativo de características</h2>

El ciclo de trabajo es:

```text
diseñar características → entrenar → medir en dev → analizar errores en dev → repetir
```

Una tentación es agregar **muchas** características de golpe. Probemos con un extractor "de fuerza bruta" que incluye todas las letras y sus conteos:

In [6]:
def caracteristicas_todo(nombre):
    nombre = nombre.lower()
    f = {"primera_letra": nombre[0], "ultima_letra": nombre[-1]}
    for letra in "abcdefghijklmnopqrstuvwxyz":
        f[f"conteo({letra})"] = nombre.count(letra)
        f[f"tiene({letra})"] = letra in nombre
    return f


_, acc_todo = evaluar(caracteristicas_todo)
print(f"Exactitud en dev (todas las letras): {acc_todo:.3f}")

Exactitud en dev (todas las letras): 0.758


Más características no siempre es mejor: con 54 características y pocos datos, el clasificador aprende patrones espurios. Es mejor **mirar los errores** y añadir características que los expliquen:

In [7]:
errores = [(g, clf_v1.classify(caracteristicas_v1(n)), n) for n, g in dev_nombres
           if clf_v1.classify(caracteristicas_v1(n)) != g]
print(f"{len(errores)} errores. Algunos:")
for real, pred, nombre in sorted(errores)[:12]:
    print(f"  real={real:<10} predicho={pred:<10} {nombre}")

229 errores. Algunos:
  real=femenino   predicho=masculino  Adriaens
  real=femenino   predicho=masculino  Alison
  real=femenino   predicho=masculino  Allsun
  real=femenino   predicho=masculino  Allyn
  real=femenino   predicho=masculino  Amber
  real=femenino   predicho=masculino  Anet
  real=femenino   predicho=masculino  Ann
  real=femenino   predicho=masculino  Ashlen
  real=femenino   predicho=masculino  Astrid
  real=femenino   predicho=masculino  Blair
  real=femenino   predicho=masculino  Brett
  real=femenino   predicho=masculino  Carlyn


Varios errores son nombres femeninos terminados en *-yn* (*Allyn*, *Carlyn*) o *-en* (*Ashlen*): la última letra *n* se asocia con nombres masculinos, pero los sufijos de dos letras cuentan otra historia. Añadimos sufijos de una, dos y tres letras:

In [8]:
def caracteristicas_v2(nombre):
    nombre = nombre.lower()
    return {
        "sufijo1": nombre[-1:],
        "sufijo2": nombre[-2:],
        "sufijo3": nombre[-3:],
        "primera_letra": nombre[0],
        "longitud": min(len(nombre), 9),
    }


clf_v2, acc_v2 = evaluar(caracteristicas_v2)
pd.DataFrame({
    "características": ["última letra", "todas las letras", "sufijos + primera letra + longitud"],
    "exactitud en dev": [acc_v1, acc_todo, acc_v2],
}).round(3)

,características,exactitud en dev
0,última letra,0.771
1,todas las letras,0.758
2,sufijos + primera letra + longitud,0.819


In [9]:
clf_v2.show_most_informative_features(10)

Most Informative Features
                 sufijo2 = 'na'           femeni : mascul =     80.2 : 1.0
                 sufijo2 = 'la'           femeni : mascul =     57.4 : 1.0
                 sufijo3 = 'ard'          mascul : femeni =     35.8 : 1.0
                 sufijo2 = 'ta'           femeni : mascul =     34.7 : 1.0
                 sufijo2 = 'us'           mascul : femeni =     31.9 : 1.0
                 sufijo2 = 'ia'           femeni : mascul =     31.4 : 1.0
                 sufijo2 = 'rd'           mascul : femeni =     30.6 : 1.0
                 sufijo1 = 'a'            femeni : mascul =     29.8 : 1.0
                 sufijo1 = 'k'            mascul : femeni =     26.2 : 1.0
                 sufijo2 = 'sa'           femeni : mascul =     24.8 : 1.0


Una vez decidido el extractor, lo evaluamos **una sola vez** en prueba:

In [10]:
test_set = [(caracteristicas_v2(n), g) for n, g in test_nombres]
print(f"Exactitud en prueba: {accuracy(clf_v2, test_set):.3f}")

Exactitud en prueba: 0.792


<a id="clasificador-de-maxima-entropia"></a>
<h2 style="color: #007ACC;">Clasificador de máxima entropía</h2>

NLTK incluye un **clasificador de máxima entropía** (*MaxEnt*). Es el mismo modelo que la **regresión logística multinomial**: para cada clase $c$ calcula un puntaje lineal sobre las características y lo convierte en probabilidad con *softmax*:

$$P(c \mid x) = \frac{\exp\left(\sum_j w_{c,j}\, f_j(x)\right)}{\sum_{c'} \exp\left(\sum_j w_{c',j}\, f_j(x)\right)}$$

El nombre viene del **principio de máxima entropía**: entre todas las distribuciones de probabilidad que reproducen los promedios de las características observados en los datos de entrenamiento, se elige la que tiene **mayor entropía**, es decir, la que hace menos suposiciones adicionales. Esa distribución resulta ser exactamente la de la fórmula anterior.

A diferencia de Naive Bayes, MaxEnt **no supone independencia** entre características: si `sufijo1=a` y `sufijo2=na` son redundantes, reparte el peso entre ellas en lugar de contar la evidencia dos veces. Esto será clave en el cuaderno 10 para los **MEMM**, que usan muchas características solapadas.

La implementación de NLTK está escrita en Python puro y es lenta, así que limitamos las iteraciones:

In [11]:
inicio = time.perf_counter()
clf_me, acc_me = evaluar(caracteristicas_v2, MaxentClassifier, algorithm="iis", max_iter=15, trace=0)
print(f"MaxEnt — exactitud en dev: {acc_me:.3f} (entrenado en {time.perf_counter() - inicio:.1f} s)")
print(f"Naive Bayes — exactitud en dev: {acc_v2:.3f}")

MaxEnt — exactitud en dev: 0.815 (entrenado en 0.8 s)
Naive Bayes — exactitud en dev: 0.819


In [12]:
clf_me.show_most_informative_features(10)

   6.057 sufijo3=='zra' and label is 'masculino'
   5.374 sufijo3=='hna' and label is 'masculino'
   4.556 sufijo3=='ous' and label is 'femenino'
   4.088 sufijo3=='vig' and label is 'femenino'
  -3.931 sufijo1=='f' and label is 'femenino'
  -3.924 sufijo1=='k' and label is 'femenino'
  -3.923 sufijo2=='na' and label is 'masculino'
   3.839 sufijo3=='eza' and label is 'masculino'
   3.836 sufijo3=='gia' and label is 'masculino'
   3.775 sufijo3=='flo' and label is 'femenino'


Los pesos de MaxEnt son interpretables como en la regresión logística: un peso positivo aumenta la probabilidad de la etiqueta indicada. Observa que los pesos más grandes corresponden a sufijos **raros** (*-zra*, *-hna*): el `MaxentClassifier` de NLTK no tiene regularización, así que puede asignar pesos extremos a características que aparecen en muy pocos ejemplos. La regresión logística de scikit-learn sí incluye regularización L2.

Además del valor más probable, ambos clasificadores devuelven una **distribución de probabilidad**:

In [13]:
for nombre in ["Andrea", "Robin", "Alexis", "Kim"]:
    dist = clf_me.prob_classify(caracteristicas_v2(nombre))
    print(f"{nombre:<8} P(femenino) = {dist.prob('femenino'):.2f}")

Andrea   P(femenino) = 0.88
Robin    P(femenino) = 0.19
Alexis   P(femenino) = 0.79
Kim      P(femenino) = 0.32


<a id="clasificacion-de-oraciones-subjetividad"></a>
<h2 style="color: #007ACC;">Clasificación de oraciones: subjetividad</h2>

Pasemos a una tarea de texto: el corpus `subjectivity` (Pang & Lee, 2004) tiene 5 000 oraciones **subjetivas** (fragmentos de reseñas de películas) y 5 000 **objetivas** (fragmentos de resúmenes de tramas). Distinguirlas es útil como paso previo al análisis de sentimiento: solo las oraciones subjetivas expresan opinión.

In [14]:
oraciones = [(s, "subj") for s in subjectivity.sents(categories="subj")] + \
            [(s, "obj") for s in subjectivity.sents(categories="obj")]
random.Random(SEED).shuffle(oraciones)

for s, etiqueta in oraciones[:4]:
    print(f"[{etiqueta}] {' '.join(s)}")

corte = int(0.8 * len(oraciones))
train_or, test_or = oraciones[:corte], oraciones[corte:]

[subj] what should have been a cutting hollywood satire is instead about as fresh as last week's issue of variety .
[obj] by doing this he meets a link between the two time periods ( lang ) .
[obj] discover what seprates the sportsstyles and the spiritual style of martial arts .
[subj] a movie so bad that it quickly enters the pantheon of wreckage that includes battlefield earth and showgirls .


El extractor de características de bolsa de palabras, al estilo NLTK, marca la presencia de cada palabra de un vocabulario fijo (las 3 000 más frecuentes del **entrenamiento**, sin stopwords). Agregamos también bigramas:

In [15]:
STOP = set(stopwords.words("english"))
frec = Counter(w.lower() for s, _ in train_or for w in s if w.isalpha() and w.lower() not in STOP)
VOCAB = {w for w, _ in frec.most_common(3000)}


def caracteristicas_oracion(tokens, bigramas=False):
    palabras = [w.lower() for w in tokens]
    f = {f"contiene({w})": True for w in palabras if w in VOCAB}
    if bigramas:
        f.update({f"bigrama({a} {b})": True for a, b in zip(palabras, palabras[1:])})
    return f


print(caracteristicas_oracion(oraciones[0][0], bigramas=True))

{'contiene(cutting)': True, 'contiene(hollywood)': True, 'contiene(satire)': True, 'contiene(instead)': True, 'contiene(fresh)': True, 'contiene(last)': True, 'contiene(issue)': True, 'contiene(variety)': True, 'bigrama(what should)': True, 'bigrama(should have)': True, 'bigrama(have been)': True, 'bigrama(been a)': True, 'bigrama(a cutting)': True, 'bigrama(cutting hollywood)': True, 'bigrama(hollywood satire)': True, 'bigrama(satire is)': True, 'bigrama(is instead)': True, 'bigrama(instead about)': True, 'bigrama(about as)': True, 'bigrama(as fresh)': True, 'bigrama(fresh as)': True, 'bigrama(as last)': True, "bigrama(last week's)": True, "bigrama(week's issue)": True, 'bigrama(issue of)': True, 'bigrama(of variety)': True, 'bigrama(variety .)': True}


In [16]:
resultados = {}
for usar_bigramas in [False, True]:
    train_set = [(caracteristicas_oracion(s, usar_bigramas), e) for s, e in train_or]
    test_set = [(caracteristicas_oracion(s, usar_bigramas), e) for s, e in test_or]
    clf = NaiveBayesClassifier.train(train_set)
    nombre = "palabras + bigramas" if usar_bigramas else "palabras"
    resultados[nombre] = (clf, test_set, accuracy(clf, test_set))
    print(f"Naive Bayes ({nombre}): exactitud = {resultados[nombre][2]:.3f}")

Naive Bayes (palabras): exactitud = 0.882


Naive Bayes (palabras + bigramas): exactitud = 0.916


In [17]:
clf_subj, test_set_subj, _ = resultados["palabras + bigramas"]
clf_subj.show_most_informative_features(15)

Most Informative Features


          bigrama(. but) = True             subj : obj    =     52.5 : 1.0
     bigrama(the film's) = True             subj : obj    =     31.8 : 1.0
       bigrama(his life) = True              obj : subj   =     31.6 : 1.0
         bigrama(-- and) = True             subj : obj    =     30.4 : 1.0
          bigrama(. the) = True             subj : obj    =     29.8 : 1.0
          contiene(plan) = True              obj : subj   =     29.6 : 1.0
      contiene(daughter) = True              obj : subj   =     28.9 : 1.0
         bigrama(him to) = True              obj : subj   =     28.2 : 1.0
      contiene(discover) = True              obj : subj   =     26.9 : 1.0
     bigrama(decides to) = True              obj : subj   =     25.6 : 1.0
   contiene(fascinating) = True             subj : obj    =     25.1 : 1.0
       bigrama(group of) = True              obj : subj   =     24.3 : 1.0
      bigrama(film that) = True             subj : obj    =     23.9 : 1.0
        contiene(former) 

Los bigramas mejoran la exactitud en más de 3 puntos, y las características más informativas muestran por qué:

- Las oraciones **objetivas** (resúmenes de trama) usan vocabulario narrativo: *daughter*, *plan*, *discover*, *his life*, *decides to*, *group of*.
- Las **subjetivas** hablan de la película como obra (*the film's*) y usan palabras evaluativas (*fascinating*). También tienen marcas de estilo: *". but"* y *"-- and"* aparecen en oraciones que encadenan una opinión con un matiz, típicas de las reseñas.

<a id="evaluacion-con-nltk-metrics"></a>
<h2 style="color: #007ACC;">Evaluación con nltk.metrics</h2>

`nltk.metrics` calcula precisión, *recall* y F1 a partir de **conjuntos**: el conjunto de índices de ejemplos que realmente pertenecen a una clase y el de los que el modelo predijo en esa clase.

In [18]:
from collections import defaultdict

referencia, predicho = defaultdict(set), defaultdict(set)
reales, predichas = [], []
for i, (feats, etiqueta) in enumerate(test_set_subj):
    pred = clf_subj.classify(feats)
    referencia[etiqueta].add(i)
    predicho[pred].add(i)
    reales.append(etiqueta)
    predichas.append(pred)

pd.DataFrame({
    c: {"precisión": precision(referencia[c], predicho[c]),
        "recall": recall(referencia[c], predicho[c]),
        "F1": f_measure(referencia[c], predicho[c])}
    for c in ["subj", "obj"]
}).T.round(3)

,precisión,recall,F1
subj,0.921,0.912,0.916
obj,0.911,0.921,0.916


In [19]:
print(ConfusionMatrix(reales, predichas))

     |       s |
     |   o   u |
     |   b   b |
     |   j   j |
-----+---------+
 obj |<915> 79 |
subj |  89<917>|
-----+---------+
(row = reference; col = test)



<a id="integracion-con-scikit-learn"></a>
<h2 style="color: #007ACC;">Integración con scikit-learn</h2>

`SklearnClassifier` permite usar cualquier clasificador de scikit-learn con la API de diccionarios de NLTK: internamente convierte los diccionarios a una matriz dispersa con `DictVectorizer`. Así combinamos la comodidad del diseño de características de NLTK con la eficiencia de scikit-learn.

In [20]:
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import BernoulliNB

train_set = [(caracteristicas_oracion(s, True), e) for s, e in train_or]

comparacion = {"NLTK NaiveBayes": resultados["palabras + bigramas"][2]}
for nombre, modelo in [("sklearn BernoulliNB", BernoulliNB()),
                       ("sklearn LogisticRegression", LogisticRegression(max_iter=1000))]:
    inicio = time.perf_counter()
    clf = SklearnClassifier(modelo).train(train_set)
    comparacion[nombre] = accuracy(clf, test_set_subj)
    print(f"{nombre:<28} entrenado en {time.perf_counter() - inicio:.2f} s")

pd.Series(comparacion, name="exactitud").round(3).to_frame()

sklearn BernoulliNB          entrenado en 0.13 s


sklearn LogisticRegression   entrenado en 1.63 s


,exactitud
NLTK NaiveBayes,0.916
sklearn BernoulliNB,0.910
sklearn LogisticRegression,0.894


En este caso los modelos de Naive Bayes rinden igual o mejor que la regresión logística, que además usamos sin ajustar su regularización (`C`).

> Con las versiones recientes de scikit-learn, algunos modelos (por ejemplo `LinearSVC`, o `LogisticRegression` con `solver="liblinear"`) fallan dentro de `SklearnClassifier` porque reciben matrices con índices de 64 bits. En esos casos conviene vectorizar los diccionarios directamente con `sklearn.feature_extraction.DictVectorizer` y usar el modelo de scikit-learn sin el envoltorio.

**Resumen:**

| | NLTK | scikit-learn |
|---|------|--------------|
| Representación | Diccionarios de características con nombre | Matrices numéricas (dispersas) |
| Fortaleza | Diseño e inspección de características; prototipado | Eficiencia, variedad de modelos, validación cruzada, pipelines |
| Uso típico en este curso | Exploración y modelos de secuencias con características (MEMM) | Entrenamiento y evaluación a escala |

Los diccionarios de características serán la pieza central del cuaderno 10: en un **MEMM** el clasificador de máxima entropía recibe, para cada palabra, características de la palabra, de sus vecinas **y de la etiqueta anterior**. Antes veremos el modelo generativo clásico para etiquetar secuencias: los **modelos ocultos de Markov**.

<a id="referencias"></a>
<h2 style="color: #007ACC;">Referencias</h2>

- Bird, S., Klein, E. & Loper, E. (2009). *Natural Language Processing with Python*. O'Reilly Media. [nltk.org/book](https://www.nltk.org/book/)
- Jurafsky, D. & Martin, J. H. *Speech and Language Processing* (3.ª ed. borrador). [web.stanford.edu/~jurafsky/slp3](https://web.stanford.edu/~jurafsky/slp3/)
- Pang, B. & Lee, L. (2004). A Sentimental Education: Sentiment Analysis Using Subjectivity Summarization Based on Minimum Cuts. *ACL*.
- Berger, A., Della Pietra, S. & Della Pietra, V. (1996). A Maximum Entropy Approach to Natural Language Processing. *Computational Linguistics*, 22(1).